# OWLv2 base/16 ensemble — DIMER E2E open-vocabulary detection adaptation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/owlv2-detection-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/owlv2-detection-pipeline/blob/main/tutorials/owlv2_detection_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google%2Fowlv2--base--patch16--ensemble-ffcc4d?style=flat)](https://huggingface.co/google/owlv2-base-patch16-ensemble) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fscenic%20(owl__vit)-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/scenic/tree/main/scenic/projects/owl_vit) [![arXiv](https://img.shields.io/badge/arXiv-2306.09683-b31b1b.svg)](https://arxiv.org/abs/2306.09683)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** zero-shot (open-vocabulary, text-prompted) object detection — one image plus 1–16 free-text phrases → score-ordered boxes labelled with the phrase they matched — and bounded supervised fine-tuning of the OWLv2 class and box heads on labelled (image, phrases, boxes) records, using the pinned `google/owlv2-base-patch16-ensemble` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/owlv2_detection_pipeline/`, at revision `49800d84b0cf`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `cfd3195ba4ea9592eec887ded089f4c08eff231d` (~622 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `google/owlv2-base-patch16-ensemble` snapshot (a 620 MB `model.safetensors`; no pickle is opened anywhere), fetches the three BCCD parquet files from the Hugging Face Hub at an immutable revision (4.8 MB together; each refused on any SHA-256 or byte-count mismatch), turns the 364 blood-smear photographs into (image, phrases, boxes) records and splits them by image into 260 / 40 / 64, detects three drawn shapes through the inference contract with an input manifest and a rejection probe, scores the frozen model over the 64 held-out records (per-phrase average precision at IoU 0.5, their mean, precision and recall at the score threshold) beside an empty and a grid-prior baseline, runs a bounded fine-tuning of the class and box heads on cached image features with validation-mAP epoch selection, scores the held-out records again, re-runs six held-out records and the drawn scene with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify detection parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). A CUDA runtime is used automatically when present. Supported-runtime timing and model-backed results have not yet been recorded for this candidate; the queued Kaggle Tesla T4 clean-runtime run is the execution gate.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one zip of images plus a `boxes.csv` (`file`, `prompt`, `x0`, `y0`, `x1`, `y1`, optional `id`; one row per object, at least eight images, at most 16 distinct phrases). The records pass through the same validation, image-disjoint split, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the BCCD sample. Uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

OWLv2 is a CLIP ViT-B/16 image tower and a CLIP text tower joined by two small heads: the image is padded to a square, resized to 960 × 960 and split into 60 × 60 = 3,600 patches, each of which the **box head** (a three-layer MLP with a per-patch position bias) turns into one box and the **class head** (a projection plus a learned logit shift and scale) scores against every phrase embedding; a patch's box survives when the **sigmoid** of its best image–text logit reaches a caller-owned threshold, and is labelled with that phrase (154,966,792 parameters in all, published under the **Apache-2.0** licence). The score is **an uncalibrated sigmoid**, there is **no non-maximum suppression**, and **the threshold is a caller-owned request parameter**.

What this notebook adds to inference is **adaptation of the class and box heads on labelled (image, phrases, boxes) records**. The records are blood-smear microscopy photographs from BCCD, each with every platelet, red blood cell and white blood cell boxed by hand — an image family and a vocabulary the detector never saw, and on the queued clean-runtime run will measure the frozen model before interpreting adaptation. The question is narrow and honest: does a bounded fine-tuning of the 1,579,526-parameter heads on 260 records — the towers frozen, the DETR-style matched loss the upstream heads were trained with — move the held-out **mAP**, **per-phrase AP**, **precision** and **recall** on an image-disjoint test split past the frozen model and two **non-adapted baselines**, and what does it do to the drawn shapes the same heads detect? Nothing here is a claim about your images or your phrases: it is one seeded split of one small labelled set.

**Snapshot note:** the pinned revision ships `model.safetensors` (a 9-file manifest with the tokenizer and processor files) — no pickle is opened anywhere in this notebook. Section 3 stages and digest-verifies those files before the processor or the model is constructed. The pipeline runs in **float32 on every device**: the adapter is trained in float32 and overlays without a cast, and CPU, Tesla-class and consumer GPUs then run the same arithmetic.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned labelled box set, turn it into (image, phrases, boxes) records, validate it and split it by image without leakage; detect in a drawn scene through the public API and read the output contract correctly (an uncalibrated sigmoid, a caller-owned threshold, no non-maximum suppression, a `sample-sanity` report only against boxes you drew yourself); measure the frozen model's held-out per-phrase AP, mAP, precision and recall beside two non-adapted baselines; run a bounded fine-tuning of the heads with the Hungarian-matched focal + L1 + GIoU loss, explicit hyperparameters and validation-based epoch selection; evaluate on an image-disjoint test split; look at the adapted boxes next to the references and at what the drawn scene does after the shared heads were tuned; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** instance or semantic segmentation, tracking, OCR, captioning, image-guided (one-shot) detection, non-maximum suppression, threshold tuning (the score threshold is fixed at `DETECTION_THRESHOLD` and the IoU threshold at `IOU_THRESHOLD` for every measurement here), fine-tuning of the CLIP image or text towers, COCO-style AP averaged over IoU thresholds, evaluation on LVIS or a detection benchmark proper (only one seeded 364-record sample is scored here), and any claim that blood-cell boxes stand in for your images. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Kaggle, Python 3.12; CPU or CUDA). The default path uses CUDA automatically when present. The image tower runs `EVAL_BATCH_SIZE` records per forward at 960 × 960. Supported-runtime timing has not yet been recorded for this candidate; the queued Kaggle Tesla T4 run is the execution gate. The pinned `torch==2.14.0` install and the 620 MB checkpoint are the large downloads; the parquet files are 4.8 MB. The feature cache holds about 1.7 GB of half-precision tensors on the host for 300 records (5.5 MB each).
- **Knowledge:** basic Python, NumPy and PIL; what a bounding box in xyxy pixel coordinates is; what intersection-over-union, average precision and a precision/recall pair at one threshold measure and why 64 records from one draw give no dispersion; why a self-drawn scene is a plumbing check while a held-out split of one labelled set is a measurement of that set only.
- **Data contract:** records are `{id, image, boxes}` — `image` a PIL image (or a file decodable by Pillow) with sides within 16..4,096 px and `boxes` 1..200 entries `{prompt, box}`: a phrase of at most 48 characters (normalised like a query; at most 16 distinct phrases per dataset) and an `[x0, y0, x1, y1]` pixel box inside the image at least 1 px wide and tall. Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a dataset needs 8..5,000 records; splitting de-duplicates by decoded pixels so no image lands in two splits. BYOD accepts one zip (or directory) of images plus a `boxes.csv` in the layout named above.
- **Validation is structural, not semantic:** every image is decoded and every phrase and box checked, but nothing checks that a box outlines what its phrase names — a mislabelled set is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path reads `full/train/0000.parquet`, `full/validation/0000.parquet` and `full/test/0000.parquet` from `https://huggingface.co/datasets/keremberke/blood-cell-object-detection/resolve/<revision>/` at the immutable parquet-conversion revision `22cf1b9d…` (4.8 MB together), each pinned by SHA-256 and byte count in the carried `samples.py` and refused on any mismatch. BCCD is Public Domain (the Roboflow Universe export of 2022-11-04); nothing is redistributed by this repository.
- **External access:** the Hugging Face Hub only, to fetch the pinned `google/owlv2-base-patch16-ensemble` snapshot (~622 MB in total) at revision `cfd3195ba4ea…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `PIL` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'owlv2-detection-pipeline',
    'repository_revision': '49800d84b0cf8df6b17b7ae63f2fb02bad140c2d',
    'embedded_module': 'src/owlv2_detection_pipeline/pipeline.py',
    'embedded_modules': ['src/owlv2_detection_pipeline/pipeline.py', 'src/owlv2_detection_pipeline/metrics.py', 'src/owlv2_detection_pipeline/samples.py'],
    'module_sha256': 'dd4e75502e8572daad3669828dca3a72dfb44534e95e4b1eaade045b938da38b',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/owlv2_detection_pipeline/` @ `49800d84b0cf`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/owlv2_detection_pipeline/pipeline.py`

In [ ]:
"""Open-vocabulary (text-prompted) object detection with the pinned ``google/owlv2-base-patch16-ensemble``
checkpoint (OWLv2), plus the adaptation contract for labelled (image, phrases, boxes) records: corpus evaluation
(per-phrase AP at an IoU threshold, precision and recall at a score threshold), bounded fine-tuning of the class
and box heads on cached image features with the DETR-style matched loss, and a verified adapter artifact.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the OWLv2 architecture comes from the pinned ``transformers`` release, the
weights are SafeTensors, and no model-repository code is executed.
"""
# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

import hashlib
import json
import random
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "google/owlv2-base-patch16-ensemble"
MODEL_REVISION = "cfd3195ba4ea9592eec887ded089f4c08eff231d"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "owlv2-base-patch16-ensemble"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Threshold: the value the pinned README's usage example passes to post_process_object_detection
# (threshold=0.1). It gates a sigmoid over the best text query per image patch that is not calibrated;
# the deployment owns tuning it on its own labelled data.
DETECTION_THRESHOLD = 0.1
# The ViT-B/16 image tower sees a 960x960 padded square as 60x60 = 3600 patch tokens, each of which
# is one detection candidate, so no image can yield more than this many boxes.
MAX_DETECTIONS = 3600
# Input ceilings. The processor pads the image to a square with grey (bottom/right) and resizes it to
# 960x960 (preprocessor_config.json), so image cost is bounded; each text query is tokenised by the
# CLIP tokenizer with model_max_length 16 (padded/truncated), so a phrase longer than that is cut.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MAX_PROMPTS = 16
MAX_PROMPT_CHARS = 48
MAX_TEXT_TOKENS = 16

# Adaptation contract. The trainable part is what the upstream authors trained on top of the CLIP towers for
# detection: the text-conditioned class head (dense projection + logit shift and scale) and the box head (a
# three-layer MLP with the per-patch box bias). The image tower, the text tower, the post-merge layer norm and
# the (unused at inference) objectness head stay frozen, so the image features can be cached once per record.
WEIGHTS_FILE = "model.safetensors"
PARAMETER_COUNT = 154_966_792
HEAD_PARAMETERS = 1_579_526  # class_head 395,266 + box_head 1,184,260 (12 tensors)
NUM_PATCHES = 3600  # 60 x 60 patch tokens = one box candidate each
_TRAINABLE_PREFIXES = ("class_head.", "box_head.")
ARTIFACT_FORMAT = f"org.valcorza.{MODEL_KEY}.adapter.v1"
ARTIFACT_VERSION = 1
ADAPTER_WEIGHTS = "adapter.safetensors"
ADAPTER_MANIFEST = "manifest.json"
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
MAX_EVAL_RECORDS = 5_000
EVAL_BATCH_SIZE = 4
GRAD_CLIP = 1.0
IOU_THRESHOLD = 0.5
# DETR-style matched loss (the OWL-ViT training objective): sigmoid focal classification over every
# (patch, query) logit, L1 and generalised IoU on the matched boxes; the same weights build the matching cost.
LOSS_WEIGHTS = {"class": 2.0, "l1": 5.0, "giou": 2.0}
FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _weight_digest(root: Path) -> str | None:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        return None
    with open(manifest_path, encoding="utf-8") as handle:
        entries = json.load(handle).get("files", [])
    return next((e["sha256"] for e in entries if e["path"] == WEIGHTS_FILE), None)


def _trainable_names(model: Any) -> list[str]:
    """The class and box head tensors; the CLIP towers, the post-merge layer norm and the objectness head stay frozen."""
    return [name for name, _ in model.named_parameters() if name.startswith(_TRAINABLE_PREFIXES)]


def _check_artifact_manifest(manifest: Mapping[str, Any], artifact_dir: Path, base_sha256: str) -> None:
    """Refuse an adapter that names another base, another format or a file that does not match its digest."""
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
    base = manifest.get("base", {})
    if base.get("model_id") != MODEL_ID or base.get("revision") != MODEL_REVISION:
        raise ValueError(f"artifact was trained on {base.get('model_id')}@{base.get('revision')}, not {MODEL_ID}@{MODEL_REVISION}")
    if base.get("weight_sha256") != base_sha256:
        raise ValueError("artifact base weight digest does not match the verified snapshot")
    files = manifest.get("files") or []
    if len(files) != 1 or files[0].get("path") != ADAPTER_WEIGHTS:
        raise ValueError(f"artifact manifest must list exactly {ADAPTER_WEIGHTS}")
    weights = artifact_dir / ADAPTER_WEIGHTS
    if not weights.is_file():
        raise FileNotFoundError(f"artifact weights missing: {weights}")
    size = weights.stat().st_size
    if size != files[0].get("bytes"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: size {size} != manifest {files[0].get('bytes')}")
    digest = _sha256(weights)
    if digest != files[0].get("sha256"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: sha256 {digest} != manifest {files[0].get('sha256')}")
    names = manifest.get("tensors") or []
    if not names or any(not str(n).startswith(_TRAINABLE_PREFIXES) for n in names):
        raise ValueError("artifact tensors must all belong to the OWLv2 class and box heads")
    adapter = manifest.get("adapter") or {}
    threshold = adapter.get("threshold")
    if isinstance(threshold, bool) or not isinstance(threshold, int | float) or not 0.0 <= threshold <= 1.0:
        raise ValueError("artifact manifest must record the score threshold the adapter was selected at")
    prompts = adapter.get("prompts")
    if not isinstance(prompts, list) or not prompts or any(not isinstance(p, str) for p in prompts):
        raise ValueError("artifact manifest must record the phrase vocabulary the adapter was trained on")


def hungarian(cost: np.ndarray) -> list[tuple[int, int]]:
    """Minimum-cost assignment of every row of a rectangular cost matrix (rows <= columns) to a distinct column —
    the shortest-augmenting-path Hungarian algorithm with numpy over the column dimension. Returns (row, column)
    pairs. Used to match each reference box to one patch candidate before the loss is computed."""
    cost = np.asarray(cost, dtype=np.float64)
    if cost.ndim != 2:
        raise ValueError("cost must be a 2-D array")
    n, m = cost.shape
    if n == 0:
        return []
    if n > m:
        raise ValueError(f"cost has more rows ({n}) than columns ({m})")
    if not np.all(np.isfinite(cost)):
        raise ValueError("cost must be finite")
    inf = float("inf")
    u = np.zeros(n + 1)
    v = np.zeros(m + 1)
    p = np.zeros(m + 1, dtype=np.int64)  # p[j] = row (1-based) assigned to column j
    way = np.zeros(m + 1, dtype=np.int64)
    padded = np.empty((n + 1, m + 1))
    padded[1:, 1:] = cost
    for i in range(1, n + 1):
        p[0] = i
        j0 = 0
        minv = np.full(m + 1, inf)
        used = np.zeros(m + 1, dtype=bool)
        while True:
            used[j0] = True
            i0 = p[j0]
            cur = padded[i0] - u[i0] - v
            better = (~used) & (cur < minv)
            minv[better] = cur[better]
            way[better] = j0
            candidates = np.where(~used, minv, inf)
            j1 = int(np.argmin(candidates))
            delta = candidates[j1]
            u[p[used]] += delta
            v[used] -= delta
            minv[~used] -= delta
            j0 = j1
            if p[j0] == 0:
                break
        while True:
            j1 = way[j0]
            p[j0] = p[j1]
            j0 = j1
            if j0 == 0:
                break
    return sorted((int(p[j]) - 1, j - 1) for j in range(1, m + 1) if p[j] != 0)


def _cxcywh_to_xyxy(boxes: Any) -> Any:
    cx, cy, w, h = boxes.unbind(-1)
    import torch

    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], -1)


def _generalized_iou(a: Any, b: Any) -> Any:
    """Pairwise GIoU between xyxy boxes a (N, 4) and b (M, 4) -> (N, M)."""
    import torch

    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    lt = torch.max(a[:, None, :2], b[None, :, :2])
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]
    union = area_a[:, None] + area_b[None, :] - inter
    iou = inter / union.clamp(min=1e-9)
    lt2 = torch.min(a[:, None, :2], b[None, :, :2])
    rb2 = torch.max(a[:, None, 2:], b[None, :, 2:])
    wh2 = (rb2 - lt2).clamp(min=0)
    enclosing = (wh2[..., 0] * wh2[..., 1]).clamp(min=1e-9)
    return iou - (enclosing - union) / enclosing


def _focal_terms(logits: Any) -> tuple[Any, Any]:
    """Per-logit sigmoid focal cost of a positive and of a negative target."""
    import torch

    prob = torch.sigmoid(logits)
    positive = FOCAL_ALPHA * (1 - prob) ** FOCAL_GAMMA * torch.nn.functional.softplus(-logits)
    negative = (1 - FOCAL_ALPHA) * prob**FOCAL_GAMMA * torch.nn.functional.softplus(logits)
    return positive, negative


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def box_iou(a: Sequence[float], b: Sequence[float]) -> float:
    """Intersection-over-union of two xyxy pixel boxes; the building block for any caller-side mAP."""
    if len(a) != 4 or len(b) != 4:
        raise ValueError("boxes must be [x0, y0, x1, y1]")
    if a[2] < a[0] or a[3] < a[1] or b[2] < b[0] or b[3] < b[1]:
        raise ValueError("boxes must satisfy x0 <= x1 and y0 <= y1")
    inter_w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return float(inter / union) if union > 0 else 0.0


def format_prompts(prompts: Sequence[str]) -> list[str]:
    """Validate a list of phrases and normalise them to the OWLv2 query form: stripped, lower-cased, one
    text query per phrase (the upstream example uses "a photo of a cat"-style queries; the pipeline
    passes the caller's phrases through unchanged apart from case and whitespace)."""
    if isinstance(prompts, str) or not isinstance(prompts, Sequence):
        raise TypeError("prompts must be a list of phrases, not a single string")
    if not 1 <= len(prompts) <= MAX_PROMPTS:
        raise ValueError(f"prompt count {len(prompts)} outside 1..MAX_PROMPTS {MAX_PROMPTS}")
    cleaned: list[str] = []
    for phrase in prompts:
        if not isinstance(phrase, str):
            raise TypeError(f"prompt must be str, got {type(phrase).__name__}")
        text = " ".join(phrase.split()).strip().rstrip(".").strip().lower()
        if not text:
            raise ValueError("prompt phrases must not be empty")
        if len(text) > MAX_PROMPT_CHARS:
            raise ValueError(
                f"prompt {text[:12]!r}... is {len(text)} chars > MAX_PROMPT_CHARS {MAX_PROMPT_CHARS}"
            )
        cleaned.append(text)
    if len(set(cleaned)) != len(cleaned):
        raise ValueError("prompt phrases must be distinct after normalisation")
    return cleaned


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(name: str, value: Any) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be a number in [0, 1], got {value!r}")
    return float(value)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one PIL.Image.Image (any mode, converted to RGB) plus 1..MAX_PROMPTS free-text phrases",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "prompts": [1, MAX_PROMPTS],
    "prompt_chars": [1, MAX_PROMPT_CHARS],
    "prompt_tokens_per_query": [1, MAX_TEXT_TOKENS],
    "threshold": [0.0, 1.0],
    "max_detections": MAX_DETECTIONS,
    "preprocessing": (
        "image converted to RGB, padded to a square with grey on the bottom/right and resized to 960x960 "
        "(CLIP mean/std); phrases stripped and lower-cased into one CLIP text query each (format_prompts, "
        "16-token limit per query); returned boxes are mapped back to input pixels"
    ),
}


def _check_inputs(image: Any, prompts: Any, threshold: Any) -> tuple[Image.Image, list[str], float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``detect`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    queries = format_prompts(prompts)
    checked = _check_threshold("threshold", threshold)
    return rgb, queries, checked


def validate_inputs(
    image: Image.Image,
    prompts: Sequence[str],
    *,
    threshold: float = DETECTION_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``detect`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, queries, checked = _check_inputs(image, prompts, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (detect takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[0] if names else "image-0",
                "mode": image.mode,
                "size": list(image.size),
                "n_prompts": len(prompts),
            }
        ],
        "queries": queries,
        "threshold": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    ground_truth_boxes: Mapping[str, Sequence[float]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``ground_truth_boxes`` (phrase -> xyxy reference box) the report carries one ``box_iou``
    entry per reference as sample-sanity geometry evidence; without them the verdict is
    ``not-measurable`` and the report says what labelled data would make the task measurable.
    """
    detections = list(result["detections"])
    base = {
        "task": "zero-shot (open-vocabulary, text-prompted) object detection",
        "decision_rule": (
            "each of the 3600 image patches proposes one box labelled with its best-matching text query; "
            "the box survives when the sigmoid of that best image-text logit reaches the threshold; the "
            "score is an uncalibrated sigmoid, not a probability, and is not exclusive across queries"
        ),
        "threshold": result.get("threshold", DETECTION_THRESHOLD),
        "sample_kind": sample_kind,
        "n_detections": len(detections),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not ground_truth_boxes:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth boxes were supplied for the evaluated image",
            "needs": (
                "labelled boxes on your own images with a phrase vocabulary matching the prompts, "
                "scored per object with box_iou and aggregated into precision/recall or mean average "
                "precision at a stated IoU threshold; no such labelled set ships with this repository"
            ),
        }
    metrics = []
    for phrase, box in ground_truth_boxes.items():
        ious = [box_iou(det["box"], box) for det in detections]
        best = max(range(len(ious)), key=ious.__getitem__) if ious else None
        metrics.append(
            {
                "id": "box_iou",
                "reference": phrase,
                "value": ious[best] if best is not None else 0.0,
                "matched_label": detections[best]["label"] if best is not None else None,
                "label_matches_reference": (detections[best]["label"] == phrase)
                if best is not None
                else False,
                "estimation": "one reference box per phrase on a single scene, no dispersion estimate",
            }
        )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} reference box(es) on one tutorial sample; geometry sanity evidence, "
            "not a detection benchmark"
        ),
        "needs": (
            "a labelled box set from the deployment domain with a matching phrase vocabulary for any "
            "mean-average-precision or precision/recall claim"
        ),
    }


@dataclass
class Owlv2DetectionPipeline:
    """Text-prompted (open-vocabulary) object detection over the pinned OWLv2 base/16 ensemble checkpoint."""

    _runner: Callable[[Image.Image, list[str], float], list[dict[str, Any]]]
    device: str
    _model: Any = None
    _processor: Any = None
    weight_sha256: str | None = None
    adapter: dict[str, Any] | None = None

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Owlv2DetectionPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Owlv2ForObjectDetection, Owlv2Processor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = Owlv2Processor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = Owlv2ForObjectDetection.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, dtype=torch.float32, **kwargs
        )
        model = model.to(resolved_device).eval()
        for param in model.parameters():
            param.requires_grad_(False)
        weight_sha256 = _weight_digest(root) if (root / MANIFEST_NAME).is_file() else None
        pipe = cls(None, resolved_device, model, processor, weight_sha256, None)  # type: ignore[arg-type]

        def runner(image: Image.Image, queries: list[str], threshold: float) -> list[dict]:
            return pipe.detect_batch([image], queries, threshold=threshold, batch_size=1)[0]

        pipe._runner = runner
        return pipe

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise RuntimeError("this pipeline has no loaded model (injected runner); use from_pretrained")
        return self._model, self._processor

    def detect(
        self,
        image: Image.Image,
        prompts: Sequence[str],
        *,
        threshold: float = DETECTION_THRESHOLD,
    ) -> dict[str, Any]:
        """Detect the phrases in `prompts`; boxes are xyxy pixel coordinates in the input image."""
        rgb, queries, checked = _check_inputs(image, prompts, threshold)
        detections = self._runner(rgb, queries, checked)
        if len(detections) > MAX_DETECTIONS:
            raise RuntimeError(
                f"backend returned {len(detections)} detections > MAX_DETECTIONS {MAX_DETECTIONS}"
            )
        for det in detections:
            if set(det) != {"box", "label", "score"} or len(det["box"]) != 4 or det["label"] not in queries:
                raise RuntimeError(f"backend returned a malformed detection: {det!r}")
        return {
            "detections": sorted(detections, key=lambda d: -d["score"]),
            "queries": queries,
            "threshold": checked,
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ------------------------------------------------------------------------------------------------------
    # Adaptation contract: batched detection, corpus evaluation, bounded fine-tuning of the heads, artifacts
    # ------------------------------------------------------------------------------------------------------

    def _encode_images(self, images: Sequence[Image.Image]) -> Any:
        """The frozen image tower on a batch: the post-merge feature map (B, 60, 60, 768) in float32 on the device."""
        model, processor = self._require_model()
        import torch

        pixel_values = processor.image_processor(images=list(images), return_tensors="pt")["pixel_values"].to(self.device)
        with torch.no_grad():
            feature_map, _vision = model.image_embedder(pixel_values=pixel_values)
        return feature_map

    def _encode_queries(self, queries: Sequence[str]) -> Any:
        """The frozen text tower on the phrase vocabulary: (Q, 512) query embeddings on the device."""
        model, processor = self._require_model()
        import torch

        tokens = processor.tokenizer(list(queries), padding="max_length", max_length=MAX_TEXT_TOKENS, truncation=True, return_tensors="pt").to(self.device)
        with torch.no_grad():
            return model.owlv2.get_text_features(input_ids=tokens["input_ids"], attention_mask=tokens["attention_mask"])

    def _heads(self, feature_map: Any, query_embeds: Any) -> tuple[Any, Any]:
        """The class and box heads on (cached) image features: exactly what `Owlv2ForObjectDetection.forward`
        computes after its frozen steps (parity asserted by the model-backed tests). Returns the (B, 3600, Q)
        logits and the (B, 3600, 4) boxes as cx, cy, w, h fractions of the padded square."""
        model, _ = self._require_model()
        import torch

        feature_map = feature_map.to(self.device, torch.float32)
        b, h, w, d = feature_map.shape
        image_feats = feature_map.reshape(b, h * w, d)
        queries = query_embeds.to(self.device, torch.float32).unsqueeze(0).expand(b, -1, -1)
        mask = torch.ones(b, queries.shape[1], dtype=torch.bool, device=self.device)
        logits, _class_embeds = model.class_predictor(image_feats, queries, mask)
        boxes = model.box_predictor(image_feats, feature_map)
        return logits, boxes

    @staticmethod
    def _postprocess(logits: Any, boxes: Any, queries: Sequence[str], sizes: Sequence[tuple[int, int]], threshold: float) -> list[list[dict[str, Any]]]:
        """The pinned processor's `post_process_grounded_object_detection` for the text path: the best query per
        patch, its sigmoid as the score, boxes scaled by max(width, height) because the image was padded to a
        square (parity with the processor asserted by the model-backed tests)."""
        import torch

        scores, labels = torch.sigmoid(logits).max(-1)
        xyxy = _cxcywh_to_xyxy(boxes)
        out = []
        for k, (width, height) in enumerate(sizes):
            keep = scores[k] >= threshold
            scale = float(max(width, height))
            boxes_k = (xyxy[k][keep] * scale).cpu().tolist()
            out.append([
                {"box": [float(v) for v in box], "label": str(queries[int(label)]), "score": float(score)}
                for box, label, score in zip(boxes_k, labels[k][keep].tolist(), scores[k][keep].tolist(), strict=True)
            ])
        return out

    def detect_batch(
        self,
        images: Sequence[Image.Image],
        prompts: Sequence[str],
        *,
        threshold: float = DETECTION_THRESHOLD,
        batch_size: int = EVAL_BATCH_SIZE,
        progress: Callable[[int, int], None] | None = None,
    ) -> list[list[dict[str, Any]]]:
        """Detect the same phrases in many images, `batch_size` images per forward; one list of ``{box, label,
        score}`` (score-descending) per image, in order. With an injected runner the images go one by one through it."""
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        checked = [_check_inputs(image, prompts, threshold) for image in images]
        cut = _check_threshold("threshold", threshold)
        queries = checked[0][1] if checked else format_prompts(prompts)
        out: list[list[dict[str, Any]]] = []
        if self._model is None:
            for rgb, q, _ in checked:
                out.append(sorted(self._runner(rgb, q, cut), key=lambda d: -d["score"]))
                if progress is not None:
                    progress(len(out), len(checked))
            return out
        query_embeds = self._encode_queries(queries)
        for start in range(0, len(checked), batch_size):
            batch = [rgb for rgb, _, _ in checked[start : start + batch_size]]
            logits, boxes = self._heads(self._encode_images(batch), query_embeds)
            for dets in self._postprocess(logits.detach(), boxes.detach(), queries, [im.size for im in batch], cut):
                if len(dets) > MAX_DETECTIONS:
                    raise RuntimeError(f"backend returned {len(dets)} detections > MAX_DETECTIONS {MAX_DETECTIONS}")
                out.append(sorted(dets, key=lambda d: -d["score"]))
            if progress is not None:
                progress(len(out), len(checked))
        return out

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        prompts: Sequence[str] | None = None,
        threshold: float = DETECTION_THRESHOLD,
        iou_threshold: float = IOU_THRESHOLD,
        batch_size: int = EVAL_BATCH_SIZE,
        progress: Callable[[int, int], None] | None = None,
    ) -> dict[str, Any]:
        """Detect the phrase vocabulary (`prompts`, default: every phrase the records use) in every validated record
        and score the detections above `threshold` against the record boxes with ``metrics.detection_metrics``
        (per-phrase AP at `iou_threshold`, their mean, precision, recall and F1). Works with an injected runner too."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import detection_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        manifest = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)
        checked = manifest["records"]
        vocabulary = format_prompts(list(prompts) if prompts is not None else manifest["prompts"])
        missing = sorted(set(manifest["prompts"]) - set(vocabulary))
        if missing:
            raise ValueError(f"records use phrases outside the evaluated vocabulary: {missing}")
        started = time.perf_counter()
        predictions = self.detect_batch([r["image"] for r in checked], vocabulary, threshold=threshold, batch_size=batch_size, progress=progress)
        metrics = detection_metrics(predictions, checked, iou_threshold=iou_threshold)
        metrics.update(
            {
                "threshold": float(threshold),
                "prompts": vocabulary,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _cache(self, records: Sequence[Mapping[str, Any]], batch_size: int, progress: Callable[[int, int], None] | None = None) -> Any:
        """Run the frozen image tower once per record and keep the feature maps in half precision on the host."""
        import torch

        maps = []
        for start in range(0, len(records), batch_size):
            batch = records[start : start + batch_size]
            maps.append(self._encode_images([r["image"] for r in batch]).to("cpu", torch.float16))
            if progress is not None:
                progress(min(start + batch_size, len(records)), len(records))
        return torch.cat(maps)

    @staticmethod
    def _targets(record: Mapping[str, Any], vocabulary: Sequence[str]) -> tuple[Any, Any]:
        """Reference boxes as (class index, cx cy w h fractions of the padded square) — the box parametrisation the
        head predicts (the processor pads the image to a square on the bottom/right, so both axes divide by the
        longer side)."""
        import torch

        scale = float(max(record["image"].size))
        index = {p: i for i, p in enumerate(vocabulary)}
        classes = torch.tensor([index[b["prompt"]] for b in record["boxes"]], dtype=torch.long)
        xyxy = torch.tensor([b["box"] for b in record["boxes"]], dtype=torch.float32) / scale
        cxcywh = torch.stack([(xyxy[:, 0] + xyxy[:, 2]) / 2, (xyxy[:, 1] + xyxy[:, 3]) / 2, xyxy[:, 2] - xyxy[:, 0], xyxy[:, 3] - xyxy[:, 1]], -1)
        return classes, cxcywh

    def _matched_loss(self, logits: Any, boxes: Any, targets: Sequence[tuple[Any, Any]]) -> tuple[Any, dict[str, float]]:
        """DETR-style loss on one batch: Hungarian matching of each reference box to one patch under the class,
        L1 and GIoU costs, then sigmoid focal loss over every (patch, query) logit (matched pairs positive), L1 and
        GIoU on the matched boxes, all normalised by the number of reference boxes in the batch."""
        import torch

        device = logits.device
        n_boxes = max(1, sum(len(c) for c, _ in targets))
        class_target = torch.zeros_like(logits)
        l1_total = torch.zeros((), device=device)
        giou_total = torch.zeros((), device=device)
        for k, (classes, gt) in enumerate(targets):
            if len(classes) == 0:
                continue
            classes, gt = classes.to(device), gt.to(device)
            with torch.no_grad():
                positive, negative = _focal_terms(logits[k].detach())
                class_cost = (positive - negative)[:, classes].transpose(0, 1)  # (n_gt, patches)
                l1_cost = torch.cdist(gt, boxes[k].detach(), p=1)
                giou_cost = -_generalized_iou(_cxcywh_to_xyxy(gt), _cxcywh_to_xyxy(boxes[k].detach()))
                cost = LOSS_WEIGHTS["class"] * class_cost + LOSS_WEIGHTS["l1"] * l1_cost + LOSS_WEIGHTS["giou"] * giou_cost
            pairs = hungarian(cost.cpu().numpy())
            rows = torch.tensor([r for r, _ in pairs], device=device)
            cols = torch.tensor([c for _, c in pairs], device=device)
            class_target[k, cols, classes[rows]] = 1.0
            matched = boxes[k][cols]
            l1_total = l1_total + (matched - gt[rows]).abs().sum()
            giou_total = giou_total + (1 - torch.diagonal(_generalized_iou(_cxcywh_to_xyxy(matched), _cxcywh_to_xyxy(gt[rows])))).sum()
        positive, negative = _focal_terms(logits)
        focal = (class_target * positive + (1 - class_target) * negative).sum() / n_boxes
        l1 = l1_total / n_boxes
        giou = giou_total / n_boxes
        total = LOSS_WEIGHTS["class"] * focal + LOSS_WEIGHTS["l1"] * l1 + LOSS_WEIGHTS["giou"] * giou
        return total, {"focal": float(focal), "l1": float(l1), "giou": float(giou)}

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None,
        *,
        prompts: Sequence[str] | None = None,
        epochs: int = 8,
        lr: float = 1e-4,
        batch_size: int = 8,
        seed: int = 0,
        threshold: float = DETECTION_THRESHOLD,
        iou_threshold: float = IOU_THRESHOLD,
        progress: Callable[[Mapping[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the OWLv2 class and box heads on labelled (image, phrases, boxes) records with the
        DETR-style matched loss the upstream detection heads were trained with (Hungarian matching; sigmoid focal
        classification, L1 and GIoU box terms). The frozen image tower is run once per record under no gradient and
        its feature maps cached (half precision on the host), the frozen text tower once per phrase, so each step
        runs only the heads; the logits equal the full model's. AdamW (no weight decay), gradient clipping at
        `GRAD_CLIP`, seeded shuffling, no scheduler, no augmentation. Epoch 0 records the frozen model's validation
        rates at `threshold`; the epoch with the highest validation mAP@`iou_threshold` (the earliest on ties) is
        kept. On any exception the frozen heads are restored."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import detection_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 100:
            raise ValueError("epochs must be an int in 1..100")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 128:
            raise ValueError("batch_size must be an int in 1..128")
        if not isinstance(lr, int | float) or isinstance(lr, bool) or not 0 < lr <= 1e-2:
            raise ValueError("lr must be a number in (0, 1e-2]")
        cut = _check_threshold("threshold", threshold)
        if not 0.0 < iou_threshold <= 1.0:
            raise ValueError("iou_threshold must be in (0, 1]")
        train_manifest = validate_dataset(train)
        train_checked = train_manifest["records"]
        val_manifest = validate_dataset(val, min_records=1) if val is not None else None
        val_checked = val_manifest["records"] if val_manifest is not None else None
        vocabulary = format_prompts(list(prompts) if prompts is not None else train_manifest["prompts"])
        used = set(train_manifest["prompts"]) | (set(val_manifest["prompts"]) if val_manifest is not None else set())
        missing = sorted(used - set(vocabulary))
        if missing:
            raise ValueError(f"records use phrases outside the training vocabulary: {missing}")
        model, _processor = self._require_model()
        import torch

        started = time.perf_counter()
        names = _trainable_names(model)
        params = {name: param for name, param in model.named_parameters() if name in set(names)}
        n_trainable = sum(p.numel() for p in params.values())
        backup = {name: param.detach().clone() for name, param in params.items()}
        previous_adapter = self.adapter
        cudnn_flags = torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark
        torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
        try:
            model.eval()
            query_embeds = self._encode_queries(vocabulary)
            cache = self._cache(train_checked, EVAL_BATCH_SIZE)
            cached_val = self._cache(val_checked, EVAL_BATCH_SIZE) if val_checked is not None else None
            cache_seconds = round(time.perf_counter() - started, 3)
            targets = [self._targets(r, vocabulary) for r in train_checked]

            def score_val() -> dict[str, Any] | None:
                if val_checked is None or cached_val is None:
                    return None
                predictions = []
                with torch.no_grad():
                    for start in range(0, len(val_checked), EVAL_BATCH_SIZE):
                        logits, boxes = self._heads(cached_val[start : start + EVAL_BATCH_SIZE], query_embeds)
                        predictions.extend(self._postprocess(logits, boxes, vocabulary, [r["image"].size for r in val_checked[start : start + EVAL_BATCH_SIZE]], cut))
                m = detection_metrics(predictions, val_checked, iou_threshold=iou_threshold)
                return {k: m[k] for k in ("map50", "precision", "recall", "f1", "n", "n_predicted_boxes")}

            for name, param in model.named_parameters():
                param.requires_grad_(name in params)
            history: list[dict[str, Any]] = [{"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}]
            if progress is not None:
                progress(history[-1])
            best_epoch, best_score = 0, (history[0]["val"] or {}).get("map50", -1.0)
            best_state = {name: param.detach().clone() for name, param in params.items()}
            optimizer = torch.optim.AdamW(list(params.values()), lr=lr, weight_decay=0.0)
            rng = random.Random(seed)
            torch.manual_seed(seed)
            order = list(range(len(train_checked)))
            for epoch in range(1, epochs + 1):
                rng.shuffle(order)
                model.class_head.train()
                model.box_head.train()
                total, steps, parts = 0.0, 0, {"focal": 0.0, "l1": 0.0, "giou": 0.0}
                for start in range(0, len(order), batch_size):
                    idx = order[start : start + batch_size]
                    optimizer.zero_grad(set_to_none=True)
                    logits, boxes = self._heads(cache[idx], query_embeds)
                    loss, terms = self._matched_loss(logits, boxes, [targets[i] for i in idx])
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(list(params.values()), GRAD_CLIP)
                    optimizer.step()
                    total += float(loss.detach())
                    for key in parts:
                        parts[key] += terms[key]
                    steps += 1
                model.eval()
                entry = {"epoch": epoch, "train_loss": round(total / max(steps, 1), 5), "loss_terms": {k: round(v / max(steps, 1), 5) for k, v in parts.items()}, "val": score_val()}
                history.append(entry)
                if progress is not None:
                    progress(entry)
                score = (entry["val"] or {}).get("map50")
                if val_checked is None or (score is not None and score > best_score):
                    best_epoch, best_score = epoch, score if score is not None else best_score
                    best_state = {name: param.detach().clone() for name, param in params.items()}
            with torch.no_grad():
                for name, param in params.items():
                    param.copy_(best_state[name])
        except BaseException:
            with torch.no_grad():
                for name, param in params.items():
                    param.copy_(backup[name])
            model.eval()
            self.adapter = previous_adapter
            raise
        finally:
            for param in model.parameters():
                param.requires_grad_(False)
            model.eval()
            torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = cudnn_flags
        self.adapter = {
            "threshold": cut,
            "iou_threshold": float(iou_threshold),
            "prompts": list(vocabulary),
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "batch_size": batch_size,
            "best_epoch": best_epoch,
            "selection": "highest validation mAP at the IoU threshold" if val_checked is not None else "final epoch (no validation split)",
            "loss": f"Hungarian-matched sigmoid focal (alpha {FOCAL_ALPHA}, gamma {FOCAL_GAMMA}) + L1 + GIoU with weights {LOSS_WEIGHTS}, normalised by the reference-box count; computed on cached image features",
            "lr": float(lr),
            "seed": seed,
            "n_train": len(train_checked),
            "n_val": len(val_checked) if val_checked is not None else 0,
            "cache_seconds": cache_seconds,
            "history": history,
            "seconds": round(time.perf_counter() - started, 3),
        }
        return dict(self.adapter)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the trained tensors as safetensors plus a manifest naming the base, the digests, the threshold, the
        phrase vocabulary and the training configuration. Requires a prior `adapt`."""
        model, _processor = self._require_model()  # refuse before importing torch
        import torch
        from safetensors.torch import save_file

        if self.adapter is None:
            raise RuntimeError("nothing to save: call adapt() first")
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = list(self.adapter["trainable_names"])
        state = model.state_dict()
        tensors = {name: state[name].detach().cpu().contiguous() for name in names}
        weights = out / ADAPTER_WEIGHTS
        save_file(tensors, str(weights), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "version": ARTIFACT_VERSION,
            "base": {"model_id": MODEL_ID, "revision": MODEL_REVISION, "weight_file": WEIGHTS_FILE, "weight_sha256": self.weight_sha256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": names,
            "files": [{"path": ADAPTER_WEIGHTS, "bytes": weights.stat().st_size, "sha256": _sha256(weights)}],
            "torch": torch.__version__,
            "metadata": dict(metadata or {}),
        }
        with open(out / ADAPTER_MANIFEST, "w", encoding="utf-8") as handle:
            json.dump(manifest, handle, indent=2, ensure_ascii=False)
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Overlay a saved adapter onto this (freshly loaded) pipeline after checking its manifest, digest and exact
        tensor set. Refuses tensors outside the class and box heads."""
        model, _processor = self._require_model()  # refuse before importing safetensors
        from safetensors.torch import load_file

        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        _check_artifact_manifest(manifest, artifact, self.weight_sha256 or "")
        expected = _trainable_names(model)
        if sorted(manifest["tensors"]) != sorted(expected):
            raise ValueError("artifact tensor set does not match its recorded configuration")
        tensors = load_file(str(artifact / ADAPTER_WEIGHTS))
        if sorted(tensors) != sorted(expected):
            raise ValueError("artifact tensor names differ from the manifest")
        state = model.state_dict()
        for name, tensor in tensors.items():
            if tuple(tensor.shape) != tuple(state[name].shape):
                raise ValueError(f"artifact tensor {name} has shape {tuple(tensor.shape)}, base has {tuple(state[name].shape)}")
        model.load_state_dict({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()}, strict=False)
        model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": expected, "history": manifest.get("history", [])}
        return dict(self.adapter)

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Owlv2DetectionPipeline:
        """Check the adapter manifest against the base snapshot's recorded weight digest, load the verified base, then
        overlay the adapter (checked again, and the tensor set, before deserialising). A refused manifest never loads
        a model."""
        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        _check_artifact_manifest(manifest, artifact, _weight_digest(root) or "")
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe

**Module 2/3:** `src/owlv2_detection_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Open-vocabulary detection metrics for labelled (image, phrases, boxes) records: per-phrase average precision
at an IoU threshold (VOC all-points interpolation), their mean, and precision / recall / F1 of the detections a
score threshold lets through, plus two non-learned baselines (no boxes; a spatial prior tiled over the image).
"""
# ruff: noqa: E501  -- metric rows are kept on single lines

from __future__ import annotations

import statistics
from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import box_iou` removed — names are kernel globals defined by the carried modules

IOU_THRESHOLD = 0.5

METRIC_DEFINITIONS = {
    "map50": "mean over the phrases with at least one reference box of the per-phrase average precision at IoU >= iou_threshold (VOC all-points interpolation over the detections ranked by score across the whole set; a reference box is matched at most once, by the highest-scoring detection of its phrase that overlaps it enough)",
    "ap": "per-phrase average precision at IoU >= iou_threshold",
    "precision": "matched detections / all detections the score threshold let through (all phrases pooled)",
    "recall": "matched reference boxes / all reference boxes (all phrases pooled)",
    "f1": "harmonic mean of precision and recall",
}


def _match(predictions: Sequence[Mapping[str, Any]], references: Sequence[Mapping[str, Any]], iou_threshold: float) -> list[tuple[float, str, bool]]:
    """Greedy matching in score order within one record: (score, phrase, matched) per prediction; a reference
    box is consumed by the first (highest-scoring) prediction of its phrase that reaches the IoU threshold."""
    ordered = sorted(predictions, key=lambda p: -float(p["score"]))
    taken = [False] * len(references)
    out = []
    for pred in ordered:
        best, best_iou = None, iou_threshold
        for k, ref in enumerate(references):
            if taken[k] or ref["prompt"] != pred["label"]:
                continue
            iou = box_iou(pred["box"], ref["box"])
            if iou >= best_iou:
                best, best_iou = k, iou
        if best is not None:
            taken[best] = True
        out.append((float(pred["score"]), str(pred["label"]), best is not None))
    return out


def average_precision(matches: Sequence[tuple[float, bool]], n_reference: int) -> float:
    """VOC all-points AP from (score, matched) pairs of one phrase over the whole set."""
    if n_reference == 0:
        return 0.0
    ordered = sorted(matches, key=lambda m: -m[0])
    tp = fp = 0
    recalls, precisions = [], []
    for _score, matched in ordered:
        tp += int(matched)
        fp += int(not matched)
        recalls.append(tp / n_reference)
        precisions.append(tp / (tp + fp))
    # make precision monotone from the right, then sum the area under the step curve
    for i in range(len(precisions) - 2, -1, -1):
        precisions[i] = max(precisions[i], precisions[i + 1])
    area, previous = 0.0, 0.0
    for recall, precision in zip(recalls, precisions, strict=True):
        area += (recall - previous) * precision
        previous = recall
    return float(area)


def detection_metrics(
    predictions: Sequence[Sequence[Mapping[str, Any]]],
    records: Sequence[Mapping[str, Any]],
    *,
    iou_threshold: float = IOU_THRESHOLD,
) -> dict[str, Any]:
    """Score one list of ``{box, label, score}`` detections per record against the record's ``boxes``."""
    if len(predictions) != len(records):
        raise ValueError(f"{len(predictions)} prediction lists for {len(records)} records")
    if not 0.0 < iou_threshold <= 1.0:
        raise ValueError("iou_threshold must be in (0, 1]")
    per_phrase: dict[str, list[tuple[float, bool]]] = {}
    n_reference: dict[str, int] = {}
    rows = []
    total_pred = total_tp = total_ref = 0
    for preds, record in zip(predictions, records, strict=True):
        refs = list(record["boxes"])
        for ref in refs:
            n_reference[ref["prompt"]] = n_reference.get(ref["prompt"], 0) + 1
        matched = _match(preds, refs, iou_threshold)
        tp = 0
        for score, phrase, hit in matched:
            per_phrase.setdefault(phrase, []).append((score, hit))
            tp += int(hit)
        rows.append({"id": record["id"], "n_reference": len(refs), "n_predicted": len(matched), "matched": tp})
        total_pred += len(matched)
        total_tp += tp
        total_ref += len(refs)
    phrases = sorted(set(n_reference) | set(per_phrase))
    per_prompt = {}
    for phrase in phrases:
        n_ref = n_reference.get(phrase, 0)
        matches = per_phrase.get(phrase, [])
        hits = sum(1 for _s, h in matches if h)
        per_prompt[phrase] = {
            "ap": round(average_precision(matches, n_ref), 4) if n_ref else None,
            "n_reference": n_ref,
            "n_predicted": len(matches),
            "precision": round(hits / len(matches), 4) if matches else 0.0,
            "recall": round(hits / n_ref, 4) if n_ref else None,
        }
    aps = [v["ap"] for v in per_prompt.values() if v["ap"] is not None]
    precision = total_tp / total_pred if total_pred else 0.0
    recall = total_tp / total_ref if total_ref else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "n": len(records),
        "n_reference_boxes": total_ref,
        "n_predicted_boxes": total_pred,
        "n_matched": total_tp,
        "iou_threshold": float(iou_threshold),
        "map50": round(statistics.fmean(aps), 4) if aps else 0.0,
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "per_prompt": per_prompt,
        "rows": rows,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def empty_baseline(records: Sequence[Mapping[str, Any]], *, iou_threshold: float = IOU_THRESHOLD) -> dict[str, Any]:
    """No detections at all: every rate is 0 by construction (the floor)."""
    out = detection_metrics([[] for _ in records], records, iou_threshold=iou_threshold)
    out["baseline"] = "empty"
    return out


def grid_baseline(
    records: Sequence[Mapping[str, Any]], reference: Sequence[Mapping[str, Any]], *, iou_threshold: float = IOU_THRESHOLD
) -> dict[str, Any]:
    """A spatial prior with no image content: for each phrase the median box size in `reference` (the training
    records) tiled edge to edge over every image, all with the same score. Beating it shows the detector reads the
    image rather than the label statistics."""
    sizes: dict[str, list[tuple[float, float]]] = {}
    for record in reference:
        for b in record["boxes"]:
            x0, y0, x1, y1 = b["box"]
            sizes.setdefault(b["prompt"], []).append((x1 - x0, y1 - y0))
    prior = {p: (statistics.median(w for w, _ in v), statistics.median(h for _, h in v)) for p, v in sizes.items()}
    predictions = []
    for record in records:
        width, height = record["image"].size
        preds = []
        for phrase, (bw, bh) in prior.items():
            x = 0.0
            while x < width:
                y = 0.0
                while y < height:
                    preds.append({"box": [x, y, min(x + bw, width), min(y + bh, height)], "label": phrase, "score": 1.0})
                    y += bh
                x += bw
        predictions.append(preds)
    out = detection_metrics(predictions, records, iou_threshold=iou_threshold)
    out["baseline"] = "grid prior (median training box per phrase tiled over the image)"
    out["prior_box_sizes"] = {p: [round(w, 1), round(h, 1)] for p, (w, h) in prior.items()}
    return out

**Module 3/3:** `src/owlv2_detection_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled (image, phrases, boxes) datasets for the adaptation contract: the digest-pinned BCCD sample, the
record contract and its structural validation, image-disjoint splitting, and the BYOD loader.

A record is ``{id, image, boxes}`` where ``image`` is a PIL image (sides within the pipeline's ceilings) and
``boxes`` a non-empty list of ``{prompt, box}``: the phrase that names the object (normalised like
``format_prompts`` normalises a query) and its ``[x0, y0, x1, y1]`` pixel box inside the image with positive
width and height. One record may carry many objects and many phrases; the phrase vocabulary of a dataset is the
sorted set of its prompts.

The default sample is BCCD (the Blood Cell Count and Detection dataset; **Public Domain**; 364 blood-smear
microscopy photographs at 416×416 with 4,886 boxes over three cell types) as exported by Roboflow to the Hugging
Face Hub (``keremberke/blood-cell-object-detection``) and converted to parquet by the Hub at an immutable revision:
the three parquet files (4.8 MB together) are fetched whole, each refused unless its SHA-256 and byte count match
the pin, and pooled into one corpus that the pipeline splits itself. The class names become the phrases
(``a platelet``, ``a red blood cell``, ``a white blood cell``). The domain gap is the point of the sample: the
queued clean-runtime run measures the frozen detector before asking what the class and box heads can learn from a
few hundred labelled microscopy images; this source-only candidate does not assume the result.
"""
# ruff: noqa: E501  -- record and pin literals are kept on single lines

from __future__ import annotations

import csv
import hashlib
import io
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import MODEL_ID, format_prompts, validate_image` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "BCCD blood-cell detection (Roboflow export), all three splits pooled"
CORPUS_REPO = "keremberke/blood-cell-object-detection"
CORPUS_REVISION = "22cf1b9d2367e799ab54a16774a2266e4f8ce9a9"  # refs/convert/parquet commit on the Hub
CORPUS_CONFIG = "full"
CORPUS_LICENSE = "Public Domain (BCCD; Roboflow Universe 'blood-cell-detection-1ekwu' export of 2022-11-04)"
# path in the conversion tree -> (sha256, bytes, rows); the whole file is fetched and refused on any mismatch
CORPUS_FILES: dict[str, tuple[str, str, int, int]] = {
    "train": (f"{CORPUS_CONFIG}/train/0000.parquet", "4079046b96b0ba20d5b1f3f6b45293a8d463ff1efa8c0c60650f5233f04ca832", 3_378_661, 255),
    "validation": (f"{CORPUS_CONFIG}/validation/0000.parquet", "0bdced43e91fe6087d3ab381d04181c57d02137d1df90ea3ede8d8eb2b4bb737", 964_704, 73),
    "test": (f"{CORPUS_CONFIG}/test/0000.parquet", "dfbd812e2c5a18592c2cc56d3c610d9cd31e29271876b2b9dd51943cc6308d08", 476_944, 36),
}
CORPUS_ROWS = 364
CORPUS_BYTES = 4_820_309
CORPUS_URL = f"https://huggingface.co/datasets/{CORPUS_REPO}/resolve/{CORPUS_REVISION}/"
DEFAULT_CACHE_DIR = Path("weights") / "bccd"

BCCD_CLASSES = ("platelets", "rbc", "wbc")  # the dataset's label ids 0, 1, 2
CLASS_PHRASES = {"platelets": "a platelet", "rbc": "a red blood cell", "wbc": "a white blood cell"}
SAMPLE_PHRASES = tuple(CLASS_PHRASES[c] for c in BCCD_CLASSES)

SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 260, "validation": 40, "test": 64}  # of the 364 pooled images
SAMPLE_DIGEST = "42d3b5df2df90c79e711f3d302dc46948bdb104f79d4cf81575221efa88d5fb2"
MIN_RECORDS = 8
MAX_RECORDS = 5_000
MAX_BOXES_PER_RECORD = 200
MIN_BOX_SIDE = 1.0
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def fetch_corpus(
    *,
    cache_dir: str | Path | None = None,
    splits: Sequence[str] | None = None,
    opener: Any = None,
) -> dict[str, bytes]:
    """Fetch the pinned parquet files (cached under `cache_dir`); every file is refused unless its SHA-256 and byte
    count match `CORPUS_FILES`. Returns the parquet bytes per source split. `opener(url) -> bytes` can replace the
    HTTPS fetch (tests inject it)."""
    root = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    root.mkdir(parents=True, exist_ok=True)
    wanted = list(splits) if splits is not None else list(CORPUS_FILES)
    out: dict[str, bytes] = {}
    for split in wanted:
        if split not in CORPUS_FILES:
            raise ValueError(f"unknown corpus split {split!r}; choose from {list(CORPUS_FILES)}")
        path, sha, size, _rows = CORPUS_FILES[split]
        cached = root / f"{split}.parquet"
        data = cached.read_bytes() if cached.is_file() else None
        if data is None or len(data) != size or _sha256_bytes(data) != sha:
            if opener is not None:
                data = opener(CORPUS_URL + path)
            else:
                request = urllib.request.Request(CORPUS_URL + path, headers={"User-Agent": "owlv2-detection-pipeline (DIMER sample fetch)"})
                with urllib.request.urlopen(request, timeout=120) as response:
                    data = response.read()
            if len(data) != size:
                raise ValueError(f"{path}: {len(data)} bytes, pinned {size}")
            digest = _sha256_bytes(data)
            if digest != sha:
                raise ValueError(f"{path}: sha256 {digest} != pinned {sha}")
            cached.write_bytes(data)
        out[split] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> list[dict[str, Any]]:
    """Decode the parquet files into records ``{id, image, boxes, source_split, source_image_id}``; COCO
    ``[x, y, w, h]`` boxes become ``[x0, y0, x1, y1]`` clipped to the image, class ids become the phrases."""
    import pyarrow.parquet as pq

    records: list[dict[str, Any]] = []
    for split, data in files.items():
        if split not in CORPUS_FILES:
            raise ValueError(f"unknown corpus split {split!r}")
        table = pq.read_table(io.BytesIO(data))
        expected_rows = CORPUS_FILES[split][3]
        if table.num_rows != expected_rows:
            raise ValueError(f"{split}: {table.num_rows} rows, pinned {expected_rows}")
        for i in range(table.num_rows):
            image_id = int(table.column("image_id")[i].as_py())
            image = Image.open(io.BytesIO(table.column("image")[i].as_py()["bytes"]))
            image.load()
            image = image.convert("RGB")
            objects = table.column("objects")[i].as_py()
            boxes = []
            for category, (x, y, w, h) in zip(objects["category"], objects["bbox"], strict=True):
                x0, y0 = max(0.0, float(x)), max(0.0, float(y))
                x1, y1 = min(float(image.width), float(x) + float(w)), min(float(image.height), float(y) + float(h))
                if x1 - x0 < MIN_BOX_SIDE or y1 - y0 < MIN_BOX_SIDE:
                    continue
                boxes.append({"prompt": CLASS_PHRASES[BCCD_CLASSES[int(category)]], "box": [x0, y0, x1, y1]})
            if not boxes:
                continue
            records.append({"id": f"bccd-{split}-{image_id}", "image": image, "boxes": boxes, "source_split": split, "source_image_id": image_id})
    return records


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]], *, seed: int = SAMPLE_SEED, sizes: Mapping[str, int] = SAMPLE_SPLIT
) -> dict[str, list[dict[str, Any]]]:
    """Seeded, image-disjoint draw of `sizes` records per split from the pooled corpus (the Roboflow split
    membership is kept only as provenance)."""
    checked = validate_dataset(records, max_records=MAX_RECORDS)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = image_digest(record["image"])
        if key not in seen:
            seen.add(key)
            unique.append(record)
    if len(unique) < sum(sizes.values()):
        raise ValueError(f"{len(unique)} distinct images < {sum(sizes.values())} requested")
    random.Random(seed).shuffle(unique)
    out: dict[str, list[dict[str, Any]]] = {}
    start = 0
    for name in ("train", "validation", "test"):
        out[name] = unique[start : start + sizes[name]]
        start += sizes[name]
    return out


def fetch_sample_dataset(*, cache_dir: str | Path | None = None, seed: int = SAMPLE_SEED) -> dict[str, list[dict[str, Any]]]:
    """The default sample: fetch (or reuse) the pinned parquet files and draw the seeded splits."""
    return build_sample_dataset(read_corpus(fetch_corpus(cache_dir=cache_dir)), seed=seed)


def _open(image: Any, where: str) -> Image.Image:
    if isinstance(image, Image.Image):
        return image
    if isinstance(image, str | Path):
        try:
            loaded = Image.open(image)
            loaded.load()
            return loaded
        except Exception as exc:  # noqa: BLE001
            raise ValueError(f"{where}: cannot decode image {image!r}") from exc
    raise ValueError(f"{where}: image must be a PIL image or a path")


def coerce_box(box: Any, size: tuple[int, int], where: str = "box") -> list[float]:
    """A ``[x0, y0, x1, y1]`` pixel box inside an image of `size` (width, height) with positive extent."""
    if isinstance(box, str | bytes) or not isinstance(box, Sequence) or len(box) != 4:
        raise ValueError(f"{where} must be [x0, y0, x1, y1]")
    try:
        x0, y0, x1, y1 = (float(v) for v in box)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{where} must hold four numbers") from exc
    if any(v != v for v in (x0, y0, x1, y1)):  # NaN
        raise ValueError(f"{where} must hold four finite numbers")
    width, height = size
    if x0 < 0 or y0 < 0 or x1 > width or y1 > height:
        raise ValueError(f"{where} [{x0}, {y0}, {x1}, {y1}] lies outside the {width}x{height} image")
    if x1 - x0 < MIN_BOX_SIDE or y1 - y0 < MIN_BOX_SIDE:
        raise ValueError(f"{where} must be at least {MIN_BOX_SIDE} px wide and tall")
    return [x0, y0, x1, y1]


def _check_record(record: Any, index: int) -> dict[str, Any]:
    where = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{where} must be a mapping with id/image/boxes")
    for key in ("id", "image", "boxes"):
        if key not in record:
            raise ValueError(f"{where} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{where}: id must match {_ID_RE.pattern}")
    try:
        image = validate_image(_open(record["image"], f"{where}.image"))
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{where}: {exc}") from exc
    boxes = record["boxes"]
    if isinstance(boxes, Mapping | str | bytes) or not isinstance(boxes, Sequence) or not 1 <= len(boxes) <= MAX_BOXES_PER_RECORD:
        raise ValueError(f"{where}: boxes must be a list of 1..{MAX_BOXES_PER_RECORD} {{prompt, box}} entries")
    checked_boxes = []
    for k, entry in enumerate(boxes):
        if not isinstance(entry, Mapping) or "prompt" not in entry or "box" not in entry:
            raise ValueError(f"{where}.boxes[{k}] must be a mapping with prompt and box")
        if not isinstance(entry["prompt"], str):
            raise ValueError(f"{where}.boxes[{k}]: prompt must be a str")
        try:
            prompt = format_prompts([entry["prompt"]])[0]
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{where}.boxes[{k}]: {exc}") from exc
        checked_boxes.append({"prompt": prompt, "box": coerce_box(entry["box"], image.size, f"{where}.boxes[{k}].box")})
    item = {"id": rid, "image": image, "boxes": checked_boxes}
    for key in ("source_split", "source_image_id"):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a labelled-box dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, boxes} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked, ids = [], set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        checked.append(item)
    counts: dict[str, int] = {}
    areas = []
    per_record = []
    for r in checked:
        per_record.append(len(r["boxes"]))
        for b in r["boxes"]:
            counts[b["prompt"]] = counts.get(b["prompt"], 0) + 1
            x0, y0, x1, y1 = b["box"]
            areas.append((x1 - x0) * (y1 - y0) / (r["image"].width * r["image"].height))
    widths = [r["image"].width for r in checked]
    heights = [r["image"].height for r in checked]
    prompts = sorted(counts)
    if len(prompts) > 16:
        raise ValueError(f"{len(prompts)} distinct prompts; the detector takes at most 16 queries per image")
    return {
        "records": checked,
        "n_records": len(checked),
        "n_boxes": sum(per_record),
        "n_prompts": len(prompts),
        "prompts": prompts,
        "boxes_per_prompt": counts,
        "boxes_per_record": {"min": min(per_record), "max": max(per_record), "mean": sum(per_record) / len(per_record)},
        "box_area_fraction": {"min": min(areas), "max": max(areas), "mean": sum(areas) / len(areas)},
        "image_width": {"min": min(widths), "max": max(widths)},
        "image_height": {"min": min(heights), "max": max(heights)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def image_digest(image: Image.Image) -> str:
    """SHA-256 of the decoded RGB pixels (size-prefixed) — the identity a split is made disjoint on."""
    rgb = image.convert("RGB")
    return _sha256_bytes(f"{rgb.width}x{rgb.height}:".encode() + rgb.tobytes())


def boxes_digest(boxes: Sequence[Mapping[str, Any]]) -> str:
    parts = sorted(f"{b['prompt']}:" + ",".join(f"{float(v):.2f}" for v in b["box"]) for b in boxes)
    return _sha256_bytes("|".join(parts).encode("utf-8"))


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """Order-independent SHA-256 over (id, image digest, boxes digest)."""
    parts = sorted(f"{r['id']}:{image_digest(r['image'])}:{boxes_digest(r['boxes'])}" for r in records)
    return _sha256_bytes("\n".join(parts).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image (by decoded-pixel digest) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]], *, val_fraction: float = 0.15, test_fraction: float = 0.2, seed: int = 0
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test after de-duplicating images."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = image_digest(record["image"])
        if key not in seen:
            seen.add(key)
            unique.append(record)
    random.Random(seed).shuffle(unique)
    n = len(unique)
    n_test = max(1, round(n * test_fraction))
    n_val = round(n * val_fraction)
    if n - n_test - n_val < 1:
        raise ValueError(f"{n} distinct images are too few to split into train/validation/test")
    return {"test": unique[:n_test], "validation": unique[n_test : n_test + n_val], "train": unique[n_test + n_val :]}


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Records from a directory or zip holding images and a `boxes.csv` with one row per object: the columns
    `file`, `prompt`, `x0`, `y0`, `x1`, `y1` (and optionally `id`); every listed file must exist and every image
    file must be listed at least once. Rows of one file become that record's boxes."""
    source = Path(path)
    members: dict[str, bytes] = {}
    if source.is_dir():
        for file in sorted(source.rglob("*")):
            if file.is_file():
                if file.name in members:
                    raise ValueError(f"BYOD data has duplicate basename {file.name!r}; use unique flat names")
                members[file.name] = file.read_bytes()
    elif zipfile.is_zipfile(source):
        with zipfile.ZipFile(source) as archive:
            for info in archive.infolist():
                if not info.is_dir():
                    name = Path(info.filename).name
                    if name in members:
                        raise ValueError(f"BYOD data has duplicate basename {name!r}; use unique flat names")
                    members[name] = archive.read(info)  # flattened; no extractall
    else:
        raise ValueError(f"{source} is neither a directory nor a zip file")
    if "boxes.csv" not in members:
        raise ValueError("BYOD data must include boxes.csv with the columns file, prompt, x0, y0, x1, y1")
    rows = list(csv.DictReader(io.StringIO(members["boxes.csv"].decode("utf-8-sig"))))
    columns = ("file", "prompt", "x0", "y0", "x1", "y1")
    if not rows or any(column not in rows[0] for column in columns):
        raise ValueError("boxes.csv must have the columns file, prompt, x0, y0, x1, y1")
    grouped: dict[str, dict[str, Any]] = {}
    for row in rows:
        name = Path(str(row.get("file", "")).strip()).name
        if name not in members:
            raise ValueError(f"boxes.csv names a missing file: {name}")
        if name not in grouped:
            try:
                image = Image.open(io.BytesIO(members[name]))
                image.load()
            except Exception as exc:  # noqa: BLE001
                raise ValueError(f"BYOD file is not a decodable image: {name}") from exc
            rid = str(row.get("id", "") or "").strip()
            grouped[name] = {"id": rid or re.sub(r"[^A-Za-z0-9_.:-]", "_", Path(name).stem)[:64], "image": image.convert("RGB"), "boxes": []}
        grouped[name]["boxes"].append({"prompt": str(row.get("prompt", "")), "box": [row.get(k, "") for k in ("x0", "y0", "x1", "y1")]})
    unlisted = [n for n in members if n != "boxes.csv" and n not in grouped]
    if unlisted:
        raise ValueError(f"{len(unlisted)} file(s) have no boxes.csv row, e.g. {unlisted[0]}")
    return list(grouped.values())


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """One row per object in the BYOD `boxes.csv` column layout plus extras (`file` names the id; the images
    themselves are not written)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "file", "prompt", "x0", "y0", "x1", "y1", "width", "height", "source_split", "source_image_id"])
        for r in records:
            for b in r["boxes"]:
                writer.writerow([r["id"], f"{r['id']}.jpg", b["prompt"], *[round(float(v), 2) for v in b["box"]], r["image"].width, r["image"].height, r.get("source_split", ""), r.get("source_image_id", "")])
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `9`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `cfd3195ba4ea…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Owlv2DetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "owlv2-base-patch16-ensemble",
  "modelId": "google/owlv2-base-patch16-ensemble",
  "revision": "cfd3195ba4ea9592eec887ded089f4c08eff231d",
  "files": [
    {
      "path": "README.md",
      "bytes": 4838,
      "sha256": "7c7426bc5ec939a42d1f96fb093031b6263400cceac4129ebb941a0c8c11b9b9"
    },
    {
      "path": "added_tokens.json",
      "bytes": 67,
      "sha256": "e5dc0da35d20111e8ff3fdfc03682beca23d5f94ed74331bce81786b2636a24f"
    },
    {
      "path": "config.json",
      "bytes": 414,
      "sha256": "ba9df8c25a4b8461887dd0a93d9252c9cd84697fe8d49a9d8794ce409af9acb2"
    },
    {
      "path": "merges.txt",
      "bytes": 524619,
      "sha256": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a"
    },
    {
      "path": "model.safetensors",
      "bytes": 619918824,
      "sha256": "e1e130b9e404cf91a75ad45644c1da9d7fa5284085eecc864266a6923efb99e7"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 425,
      "sha256": "cf3e396635b797ee1a464e1b2836e98748f8edac19e89aaa2c93b55ac15b0064"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 121,
      "sha256": "d6e2b9cf664efbad2d22998b8d3da986abcbeed3e0825ad33605c9401f9cf73e"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 1100,
      "sha256": "b55cda6198e152ded427c8a9b3faf1cccf27a7fa080697a62f6ff143f511f44f"
    },
    {
      "path": "vocab.json",
      "bytes": 1059962,
      "sha256": "e089ad92ba36837a0d31433e555c8f45fe601ab5c221d4f607ded32d9f7a4349"
    }
  ],
  "totalBytes": 621510370
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Owlv2DetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. BCCD records, the phrases and the split

`fetch_corpus` returns the three pinned parquet files from the cache under `weights/bccd/` or the Hub at the pinned parquet-conversion revision — every cached file is re-hashed and every fetched file refused on any SHA-256 or byte-count mismatch — and `read_corpus` turns each row into a record: the photograph and one `{prompt, box}` per annotated cell, the COCO `[x, y, w, h]` boxes converted to `[x0, y0, x1, y1]` and the three class ids to the phrases `a platelet`, `a red blood cell`, `a white blood cell`. The Roboflow split membership is kept only as provenance: `build_sample_dataset` pools the 364 images and draws a seeded image-level split (260 / 40 / 64). `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no image (by decoded-pixel digest) is shared, and the training split's box table is written to `outputs/owlv2_detection_train.csv`.

Look for: 364 records at 416 × 416 with about thirteen boxes each (red blood cells dominate: roughly eleven per image, one white blood cell, one platelet), three digests, and four refusal probes — a duplicate id, a box outside its image, a record without boxes, and a dataset too small to use — each rejected before the model does anything.

In [ ]:
import hashlib
import json
import time

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_zip = Path('work') / 'byod.zip'
    byod_zip.parent.mkdir(parents=True, exist_ok=True)
    byod_zip.write_bytes(payload)
    records = load_byod_dataset(byod_zip)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    t0 = time.perf_counter()
    corpus_files = fetch_corpus(cache_dir='weights/bccd')
    corpus = read_corpus(corpus_files)
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} @ {CORPUS_REVISION[:12]} ({CORPUS_LICENSE})'
    raw_rows = {'files': sorted(corpus_files), 'bytes': sum(len(v) for v in corpus_files.values()), 'records': len(corpus), 'boxes': sum(len(r['boxes']) for r in corpus), 'seconds': round(time.perf_counter() - t0, 1)}
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
PROMPTS = sorted(set(dataset_manifests['train']['prompts']) | set(dataset_manifests['validation']['prompts']) | set(dataset_manifests['test']['prompts']))
write_dataset_csv(train_records, 'outputs/owlv2_detection_train.csv')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'prompts': PROMPTS})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'boxes': manifest['n_boxes'], 'boxes_per_prompt': manifest['boxes_per_prompt'], 'boxes_per_record': {k: round(v, 1) for k, v in manifest['boxes_per_record'].items()}, 'box_area': {k: round(v, 3) for k, v in manifest['box_area_fraction'].items()}, 'width': manifest['image_width'], 'height': manifest['image_height'], 'digest': manifest['digest'][:16] + '...'}})


PALETTE = {p: c for p, c in zip(PROMPTS, [(220, 40, 40), (40, 70, 200), (60, 179, 75), (250, 200, 30), (160, 60, 200), (0, 170, 170)] * 3, strict=False)}


def draw_boxes(image, boxes, width=2, key='prompt', dashed=False):
    """Outline each {prompt|label, box} on a copy of the image in the phrase's colour."""
    out = image.convert('RGB').copy()
    d = ImageDraw.Draw(out)
    for b in boxes:
        colour = PALETTE.get(b[key], (20, 20, 20))
        x0, y0, x1, y1 = b['box']
        if dashed:
            for x in np.arange(x0, x1, 6):
                d.line([(x, y0), (min(x + 3, x1), y0)], fill=colour, width=width)
                d.line([(x, y1), (min(x + 3, x1), y1)], fill=colour, width=width)
            for y in np.arange(y0, y1, 6):
                d.line([(x0, y), (x0, min(y + 3, y1))], fill=colour, width=width)
                d.line([(x1, y), (x1, min(y + 3, y1))], fill=colour, width=width)
        else:
            d.rectangle([x0, y0, x1, y1], outline=colour, width=width)
    return out


example = train_records[0]
draw_boxes(example['image'], example['boxes']).save('outputs/owlv2_detection_example_record.png')
print({'example': {'id': example['id'], 'image': list(example['image'].size), 'boxes': len(example['boxes']), 'phrases': sorted({b['prompt'] for b in example['boxes']})}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'box outside the image': [{**train_records[0], 'boxes': [{'prompt': 'a platelet', 'box': [0, 0, train_records[0]['image'].width + 10, 20]}]}, *train_records[1:8]],
    'record without boxes': [{**train_records[0], 'boxes': []}, *train_records[1:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Detect in a drawn scene through the inference contract

The inference contract is exercised as the inference-only tutorial exercised it: a deterministic 640 × 480 scene drawn in code — a black rectangle at `[80, 120, 280, 360]`, a red disc with bounding box `[380, 140, 560, 320]` and a blue triangle with bounding box `[200, 380, 360, 460]` — with the prompts naming them and one absent phrase (`a green star`) on purpose; a different image family from the blood smears, and a scene the adapted model will detect in again in Section 9. `validate_inputs` applies exactly the checks `detect` applies (one image with sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE`, 1..`MAX_PROMPTS` distinct phrases of at most `MAX_PROMPT_CHARS` characters, a threshold in [0, 1]) and returns an input manifest; an over-long phrase is validated too and its rejection recorded as a finding. `detect` returns `{box, label, score}` detections **ordered by descending score** — **the scores are an uncalibrated sigmoid** of the best image–text logit, the threshold is a **caller-owned request parameter**, and there is **no non-maximum suppression**, so one object can surface as several boxes at a low threshold. `evaluation_report` with the drawn boxes is `sample-sanity`: one `box_iou` per drawn phrase, plumbing evidence for one drawing — a detection benchmark needs labelled boxes, which Section 6 supplies. The inference-only card recorded IoUs of 0.95–0.97 on the three drawn shapes.

In [ ]:
def synthetic_scene(width=640, height=480):
    """Three coloured shapes drawn with Pillow (no text); returns image + {phrase: xyxy reference box}."""
    image = Image.new('RGB', (width, height), (128, 128, 128))
    d = ImageDraw.Draw(image)
    boxes = {'a black rectangle': [80.0, 120.0, 280.0, 360.0], 'a red circle': [380.0, 140.0, 560.0, 320.0], 'a blue triangle': [200.0, 380.0, 360.0, 460.0]}
    d.rectangle(boxes['a black rectangle'], fill=(30, 30, 30))
    d.ellipse(boxes['a red circle'], fill=(220, 30, 30))
    d.polygon([(280, 380), (200, 460), (360, 460)], fill=(30, 60, 220))
    return image, boxes


scene, scene_boxes = synthetic_scene()
scene_prompts = list(scene_boxes) + ['a green star']  # one absent phrase on purpose
scene_name = 'synthetic_scene_640x480'
scene_sha256 = hashlib.sha256(np.asarray(scene).tobytes()).hexdigest()
THRESHOLD = DETECTION_THRESHOLD
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PROMPTS': MAX_PROMPTS, 'MAX_PROMPT_CHARS': MAX_PROMPT_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MAX_DETECTIONS': MAX_DETECTIONS, 'NUM_PATCHES': NUM_PATCHES, 'DETECTION_THRESHOLD': DETECTION_THRESHOLD, 'IOU_THRESHOLD': IOU_THRESHOLD, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'EVAL_BATCH_SIZE': EVAL_BATCH_SIZE, 'device': pipe.device}})
input_manifest = validate_inputs(scene, scene_prompts, threshold=THRESHOLD, names=[scene_name])
try:
    validate_inputs(scene, ['x' * (MAX_PROMPT_CHARS + 1)])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'over-long-prompt-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/owlv2_detection_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'scene': scene_name, 'sha256': scene_sha256[:16] + '...', 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})


def detect_scene(pipeline, label):
    started = time.perf_counter()
    result = pipeline.detect(scene, scene_prompts, threshold=THRESHOLD)
    seconds = round(time.perf_counter() - started, 3)
    checks = {
        'score_ordered': all(a['score'] >= b['score'] for a, b in zip(result['detections'], result['detections'][1:], strict=False)),
        'labels_are_queries': all(d['label'] in result['queries'] for d in result['detections']) and result['queries'] == [q.lower() for q in scene_prompts],
        'boxes_inside_the_image': all(0 <= d['box'][0] <= d['box'][2] <= scene.width + 1 and 0 <= d['box'][1] <= d['box'][3] <= scene.height + 1 for d in result['detections']),
        'identity_reported': result['model_id'] == MODEL_ID and result['model_revision'] == MODEL_REVISION,
    }
    if not all(checks.values()):
        raise RuntimeError(f'detect output failed a sanity check: {checks}')
    report = evaluation_report(result, scene_boxes, sample_kind='synthetic (drawn in this notebook)')
    ious = {m['reference']: {'iou': round(m['value'], 3), 'label_matches': m['label_matches_reference']} for m in report['metrics']}
    per_label = {q: sum(1 for d in result['detections'] if d['label'] == q) for q in result['queries']}
    top = [(d['label'], round(d['score'], 3), [round(v, 1) for v in d['box']]) for d in result['detections'][:5]]
    with open(f'outputs/owlv2_detection_scene_{label}.json', 'w', encoding='utf-8') as handle:
        json.dump({'detections': result['detections'], 'report': report}, handle, indent=2, ensure_ascii=False)
    annotated = scene.copy()
    marker = ImageDraw.Draw(annotated)
    for d in result['detections']:
        marker.rectangle(d['box'], outline=(0, 255, 0), width=2)
        marker.text((d['box'][0] + 2, d['box'][1] + 2), f"{d['label']} {d['score']:.2f}", fill=(0, 255, 0))
    annotated.save(f'outputs/owlv2_detection_scene_{label}.png')
    summary = {'n_detections': len(result['detections']), 'per_label': per_label, 'top': top, 'box_iou': ious}
    print({label: {'seconds': seconds, 'checks': checks, **summary, 'verdict': report['verdict']}})
    return summary, seconds, checks, report


frozen_scene, frozen_scene_seconds, frozen_scene_checks, frozen_scene_report = detect_scene(pipe, 'frozen')

## 6. Baselines and the frozen model on the test records

Two non-adapted baselines frame the adaptation, each scored by `detection_metrics` (carried in `metrics.py`): per phrase, the **average precision at IoU ≥ 0.5** — the detections of that phrase ranked by score across the whole split, each reference box matched at most once, the area under the precision–recall step curve (VOC all-points) — and their mean, **mAP@0.5**, the measure the epoch is selected on; and, over all phrases pooled, the **precision** and **recall** of the detections the score threshold let through and their F1. The **empty** baseline predicts no box and scores 0 by construction — the floor. The **grid-prior** baseline tiles each phrase's median training box edge to edge over every image with one constant score, what the label statistics buy without looking at the image. The **frozen model** is scored by `pipe.evaluate`, which detects the three phrases in every record in batches of `EVAL_BATCH_SIZE`, keeps the boxes whose sigmoid reaches `THRESHOLD`, and scores them. Do not assume the frozen detector's behaviour in advance: read its mAP, per-phrase APs, precision, recall, box counts and per-record rows from this run.

In [ ]:
METRICS = ('map50', 'precision', 'recall', 'f1')

baseline_empty = empty_baseline(test_records)
baseline_grid = grid_baseline(test_records, train_records)
print({'empty_baseline': {k: round(baseline_empty[k], 3) for k in METRICS}, 'n': baseline_empty['n'], 'note': baseline_empty['baseline']})
print({'grid_baseline': {k: round(baseline_grid[k], 3) for k in METRICS}, 'prior_box_sizes': baseline_grid['prior_box_sizes'], 'n_predicted_boxes': baseline_grid['n_predicted_boxes'], 'note': baseline_grid['baseline']})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, prompts=PROMPTS, threshold=THRESHOLD, batch_size=EVAL_BATCH_SIZE)
print({'frozen_model_test': {k: round(frozen_test[k], 3) for k in METRICS}, 'n': frozen_test['n'], 'reference_boxes': frozen_test['n_reference_boxes'], 'predicted_boxes': frozen_test['n_predicted_boxes'], 'matched': frozen_test['n_matched'], 'verdict': frozen_test['verdict'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'frozen_per_prompt': frozen_test['per_prompt']})
print({'definitions': frozen_test['definitions']})
for row in frozen_test['rows'][:4]:
    print(row)

## 7. Bounded fine-tuning of the heads

`pipe.adapt` trains only the two detection heads — the class head (a 768 → 512 projection with a learned logit shift and scale) and the box head (a three-layer MLP): 1,579,526 of 154,966,792 parameters — while the CLIP image and text towers, the post-merge layer norm and the objectness head stay frozen, the split the upstream authors trained with. The loss is the **DETR-style matched loss** the OWL-ViT heads were trained with: each reference box is assigned to one of the 3,600 patch candidates by **Hungarian matching** under a cost of focal classification, L1 and generalised-IoU terms, then the **sigmoid focal loss** is taken over every (patch, phrase) logit with the matched pairs as positives, and **L1 and GIoU** over the matched boxes, all normalised by the number of reference boxes. Because the towers are frozen, their outputs — the 60 × 60 × 768 feature map per image and the three phrase embeddings — are computed once under no gradient and cached in half precision on the host (the **frozen-tower cache**), and each step runs only the heads on those cached features: the logits equal the full model's. AdamW without weight decay at a fixed learning rate, gradient clipping at 1.0, seeded shuffling, no scheduler, no augmentation. Epoch 0 records the frozen model's validation rates; every epoch is scored on the 40 validation records at `THRESHOLD`, and the epoch with the **highest validation mAP** (the earliest on ties) is kept.

Read the emitted epoch history rather than assuming improvement: it records the validation mAP and loss for every epoch, then restores the earliest epoch with the highest validation mAP.

In [ ]:
EPOCHS = 8  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('loss_terms'):
        row['terms'] = {k: round(v, 3) for k, v in entry['loss_terms'].items()}
    if entry.get('val'):
        row.update({'val_' + k: round(entry['val'][k], 3) for k in METRICS})
        row['val_boxes'] = entry['val']['n_predicted_boxes']
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, prompts=PROMPTS, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, threshold=THRESHOLD, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'threshold': adapt_result['threshold'], 'iou_threshold': adapt_result['iou_threshold'], 'prompts': adapt_result['prompts'], 'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'loss': adapt_result['loss'], 'cache_seconds': adapt_result['cache_seconds'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test records were never used for training or epoch selection, and no image appears in two splits. The adapted model is scored exactly as the frozen model was in Section 6 and the four systems are put side by side. Read it in this order: **mAP@0.5** first (the measure the epoch was selected on), then the **per-phrase APs**, then **precision** and **recall** together — a gain in one at the cost of the other is a moved threshold, not a better detector), then the number of predicted boxes against the reference count. The cell asserts the adapted mAP is at least the frozen one and above the grid-prior baseline. Sixty-four records from one seeded split give **no dispersion estimate**; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a result on one blood-smear set's three cell types says nothing about other phrases, other images or your data until you measure them.

In [ ]:
adapted_test = pipe.evaluate(test_records, prompts=PROMPTS, threshold=THRESHOLD, batch_size=EVAL_BATCH_SIZE)
adapted_val = pipe.evaluate(val_records, prompts=PROMPTS, threshold=THRESHOLD, batch_size=EVAL_BATCH_SIZE)
comparison = {metric: {'empty': round(baseline_empty[metric], 3), 'grid': round(baseline_grid[metric], 3), 'frozen': round(frozen_test[metric], 3), 'adapted': round(adapted_test[metric], 3)} for metric in METRICS}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 3) for metric in METRICS}
comparison['per_prompt_ap'] = {p: {'frozen': frozen_test['per_prompt'][p]['ap'], 'adapted': adapted_test['per_prompt'][p]['ap'], 'n_reference': adapted_test['per_prompt'][p]['n_reference']} for p in PROMPTS}
comparison['boxes'] = {'reference': adapted_test['n_reference_boxes'], 'frozen_predicted': frozen_test['n_predicted_boxes'], 'adapted_predicted': adapted_test['n_predicted_boxes'], 'frozen_matched': frozen_test['n_matched'], 'adapted_matched': adapted_test['n_matched']}
for key, row in comparison.items():
    print({key: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'threshold': THRESHOLD,
    'iou_threshold': IOU_THRESHOLD,
    'prompts': PROMPTS,
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'baselines': {'empty': {k: v for k, v in baseline_empty.items() if k != 'rows'}, 'grid': {k: v for k, v in baseline_grid.items() if k != 'rows'}},
    'frozen_test': {k: v for k, v in frozen_test.items() if k != 'rows'},
    'validation_metrics': {k: v for k, v in adapted_val.items() if k != 'rows'},
    'test_metrics': {k: v for k, v in adapted_test.items() if k != 'rows'},
    'per_record': [{**frozen_row, 'adapted_n_predicted': adapted_row['n_predicted'], 'adapted_matched': adapted_row['matched']} for frozen_row, adapted_row in zip(frozen_test['rows'], adapted_test['rows'], strict=True)],
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/owlv2_detection_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['map50'] >= frozen_test['map50']
assert adapted_test['map50'] > baseline_grid['map50']
print({'report': 'outputs/owlv2_detection_evaluation_report.json', 'adapted_beats_both_baselines': adapted_test['map50'] > max(baseline_empty['map50'], baseline_grid['map50'])})

## 9. Look at the boxes, detect in the scene again, export the adapter and reload it

Six held-out records are written as panels (`outputs/owlv2_detection_examples/`: the photograph with the reference boxes, the frozen detections and the adapted detections side by side, coloured by phrase, the counts beneath) so the numbers can be checked by eye. The drawn scene from Section 5 is then detected in again by the adapted model — the heads that were tuned serve every phrase, so this is a small look at what the adaptation did *outside* its phrase vocabulary and its corpus. Treat that one drawing as qualitative evidence, not a measurement.

`pipe.save_artifact` writes the trained tensors — the two heads, about 6.3 MB in float32 — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the score and IoU thresholds the epoch was selected at, the phrase vocabulary, the training configuration and the epoch history (OUT8). `Owlv2DetectionPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor outside the two heads, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical detections on eight test records (VER4).

In [ ]:
import shutil

examples_dir = Path('outputs/owlv2_detection_examples')
shutil.rmtree(examples_dir, ignore_errors=True)
examples_dir.mkdir(parents=True)
caption_font = ImageFont.load_default(size=18)
adapted_items =pipe.detect_batch([r['image'] for r in test_records[:6]], PROMPTS, threshold=THRESHOLD)
for record, frozen_row, adapted_row, adapted_dets in zip(test_records[:6], frozen_test['rows'][:6], adapted_test['rows'][:6], adapted_items, strict=True):
    panels = [draw_boxes(record['image'], record['boxes']), draw_boxes(record['image'], adapted_dets, key='label', dashed=True)]
    sheet = Image.new('RGB', (sum(p.width for p in panels) + 8, panels[0].height + 56), (255, 255, 255))
    x = 0
    for panel in panels:
        sheet.paste(panel, (x, 0))
        x += panel.width + 8
    marker = ImageDraw.Draw(sheet)
    marker.text((8, panels[0].height + 6), f"reference {frozen_row['n_reference']} boxes (solid) | adapted {adapted_row['n_predicted']} boxes, {adapted_row['matched']} matched (dashed); frozen matched {frozen_row['matched']} of {frozen_row['n_predicted']}", fill=(20, 20, 20), font=caption_font)
    sheet.save(examples_dir / f"{record['id']}.png")
print({'examples': sorted(p.name for p in examples_dir.iterdir()), 'panels': ['reference boxes', 'adapted detections'], 'palette': PALETTE})

adapted_scene, adapted_scene_seconds, adapted_scene_checks, adapted_scene_report = detect_scene(pipe, 'adapted')

artifact_dir = Path('outputs/owlv2_detection_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'owlv2_detection', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...', 'threshold': artifact_manifest['adapter']['threshold'], 'prompts': artifact_manifest['adapter']['prompts'], 'best_epoch': artifact_manifest['adapter']['best_epoch']})

reloaded = Owlv2DetectionPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = pipe.detect_batch([r['image'] for r in test_records[:8]], PROMPTS, threshold=THRESHOLD)
after = reloaded.detect_batch([r['image'] for r in test_records[:8]], PROMPTS, threshold=THRESHOLD)


def same(dets):
    return [(d['label'], round(d['score'], 4), tuple(round(v, 1) for v in d['box'])) for d in dets]


parity ={'identical_detections': sum(bool(same(a) == same(b)) for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_detections'] == parity['of']

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHTS_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': pipe.weight_sha256},
    'data_source': data_source,
    'threshold': THRESHOLD,
    'iou_threshold': IOU_THRESHOLD,
    'prompts': PROMPTS,
    'corpus': {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'files': {k: v[0] for k, v in CORPUS_FILES.items()}, 'license': CORPUS_LICENSE, 'bytes': CORPUS_BYTES, 'rows': CORPUS_ROWS, 'classes': list(BCCD_CLASSES), 'class_phrases': CLASS_PHRASES},
    'inference_contract': {'input_manifest': input_manifest, 'scene': {'name': scene_name, 'sha256': scene_sha256, 'prompts': scene_prompts, 'drawn_boxes': scene_boxes}, 'frozen': {'summary': frozen_scene, 'seconds': frozen_scene_seconds, 'checks': frozen_scene_checks, 'report': frozen_scene_report}, 'adapted': {'summary': adapted_scene, 'seconds': adapted_scene_seconds, 'checks': adapted_scene_checks, 'report': adapted_scene_report}, 'output_files': ['outputs/owlv2_detection_scene_frozen.json', 'outputs/owlv2_detection_scene_adapted.json', 'outputs/owlv2_detection_scene_frozen.png', 'outputs/owlv2_detection_scene_adapted.png']},
    'comparison': comparison,
    'examples': 'outputs/owlv2_detection_examples',
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'pillow': PIL.__version__, 'device': pipe.device, 'dtype': 'float32'},
}
with open('outputs/owlv2_detection_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

This candidate asks whether bounded fine-tuning of an open-vocabulary detector's class and box heads on 260 labelled blood-smear records improves held-out detection relative to two non-adapted baselines and the frozen model. The clean-runtime run must answer that question; this source-only build makes no numerical claim. Read the emitted per-phrase AP at one IoU threshold, precision and recall at one score threshold, and predicted-box counts together rather than in isolation. The exported adapter is accepted only after fresh-base reload parity passes.

The test split is 64 records from one seeded draw of one 364-record sample, the validation split that picks the epoch is 40, and every rate is at the one score threshold `DETECTION_THRESHOLD` and the one IoU threshold `IOU_THRESHOLD` — not a benchmark, not a threshold sweep, not COCO-style AP averaged over IoU thresholds, not a measure of phrases the sample never asks. So a result here says the contract works on three blood-cell phrases, not that the adapted model handles other phrases, other image families or your boxes. The heads that were tuned serve every phrase: the drawn scene re-detected in Section 9 is one drawing of evidence about what the tuning did outside its vocabulary, not a measurement, and a deployment that detects other phrases must measure them after adapting. The towers were not adapted: what the image encoder cannot see stays undetected, **the scores remain an uncalibrated sigmoid**, and there is still no non-maximum suppression.

Three things to carry to real data. **Baselines first:** the empty and grid-prior rates on *your* boxes, and the frozen model's box count, are the numbers to read before any adapted one. **Precision and recall together:** a gain in mAP that comes with a collapse of one of them is a moved threshold, and the threshold is yours to set on a validation split, not the test split. **Leakage:** keep every image in one split (the contract de-duplicates by decoded pixels) and split by source, session or slide when your images come from few sources.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real labelled box set, validate the demonstrated dataset contract without leakage, execute the inference contract for a drawn scene and a bounded fine-tuning of the heads with the upstream objective, evaluate against two non-adapted baselines and the frozen model on an image-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, detection quality on any other phrase vocabulary or image family, calibration of the sigmoid, or production fitness.

**Optional experiments (they do not affect the default path):** raise `EPOCHS` and watch the validation mAP pick the epoch; change `LEARNING_RATE` by a factor of ten in either direction and read the curve; set `THRESHOLD` to `0.05` or `0.3` before Section 6 and read how precision and recall trade against each other for both the frozen and the adapted model; change `SPLIT_SEED` and read how much 64 records move; or bring your own boxes through BYOD and read the two baselines before the adapted number.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from `weights/owlv2-base-patch16-ensemble/` and rerun Section 3. A `sha256` `ValueError` naming a parquet file in Section 4: a cached `weights/bccd/*.parquet` is incomplete — delete it and rerun Section 4.

## References

- Repository README: https://github.com/kurtvalcorza/owlv2-detection-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/owlv2-detection-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/owlv2-detection-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/google/owlv2-base-patch16-ensemble
- Upstream code (Scenic, OWL-ViT project): https://github.com/google-research/scenic/tree/main/scenic/projects/owl_vit
- Scaling Open-Vocabulary Object Detection (Minderer, Gritsenko, Houlsby, 2023): https://arxiv.org/abs/2306.09683
- Simple Open-Vocabulary Object Detection with Vision Transformers (Minderer et al., 2022): https://arxiv.org/abs/2205.06230
- BCCD (Public Domain): https://huggingface.co/datasets/keremberke/blood-cell-object-detection — the Roboflow Universe export of the Blood Cell Count and Detection dataset
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)